# Smart-contract robustness and gas evaluation

This notebook evaluates adversarial and failure scenarios on the IOTA EVM testnet and exports two publication-ready CSV files. Transactions expected to revert are actually mined so their gas consumption is measured.

**Security:** use fresh test-only accounts. A private key posted in chat or committed to a notebook must be treated as compromised.

In [1]:
%pip install -q web3 pandas

import time
from decimal import Decimal
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display
from web3 import Web3


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

Enter fresh testnet keys when prompted. `ORACLE_PRIVATE_KEY` must correspond to the backend/oracle address authorized by the deployed contract. `ATTACKER_PRIVATE_KEY` must be an unauthorized test account.

In [2]:
RPC_URL = "https://json-rpc.evm.testnet.iota.cafe"
CHAIN_ID = 1076
SMART_CONTRACT_ADDRESS = "0x645aDd21578B9fc69Cfbe38bf18F2698Bb449129"

RESERVE_VALUE_WEI = Web3.to_wei(0.2, "ether")
GAS_LIMIT = 500_000
RECEIPT_TIMEOUT = 180
TEST_SLOTS = ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9", "T10"]
NONEXISTENT_ID = 2**255

#CLIENT_PRIVATE_KEY = getpass("Fresh client test key: ") # c150364d9d8633a5af50b41389b1ae388c4f3029367cee65ef289026d7d9a9a9 Acc1
#ORACLE_PRIVATE_KEY = getpass("Authorized oracle/backend test key: ") # 935064845b2be2b787c7b0716336b7f8e07e3005697088a498d82a1cb637e483 Acc3
#ATTACKER_PRIVATE_KEY = getpass("Unauthorized attacker test key: ") # 93f86ad1b2adc4b2c32b4dfc31e1f7c147c8ca66eb430809553aa011beb2d043 Acc4

## 2. Contract ABI

The following ABI assumes `confirmArrival(uint256)`. If your deployed signature differs, replace that entry and adapt `confirm_fn()` below. For the final evaluation, using the complete compiled ABI is preferable.

In [3]:
ABI = [
    {"inputs": [{"internalType": "string", "name": "slot", "type": "string"}],
     "name": "reserve", "outputs": [{"internalType": "uint256", "name": "id", "type": "uint256"}],
     "stateMutability": "payable", "type": "function"},
    {"inputs": [{"internalType": "uint256", "name": "id", "type": "uint256"}],
     "name": "confirmArrival", "outputs": [], "stateMutability": "nonpayable", "type": "function"},
    {"inputs": [{"internalType": "uint256", "name": "id", "type": "uint256"}],
     "name": "finalizeNoShow", "outputs": [], "stateMutability": "nonpayable", "type": "function"},
    {"anonymous": False, "inputs": [
        {"indexed": True, "internalType": "uint256", "name": "id", "type": "uint256"},
        {"indexed": True, "internalType": "address", "name": "user", "type": "address"},
        {"indexed": True, "internalType": "bytes32", "name": "slotHash", "type": "bytes32"},
        {"indexed": False, "internalType": "uint256", "name": "price", "type": "uint256"},
        {"indexed": False, "internalType": "uint64", "name": "createdAt", "type": "uint64"},
        {"indexed": False, "internalType": "uint64", "name": "deadline", "type": "uint64"},
        {"indexed": False, "internalType": "uint16", "name": "trustScore", "type": "uint16"}],
     "name": "ReservationCreated", "type": "event"},
    {"anonymous": False, "inputs": [
        {"indexed": True, "internalType": "uint256", "name": "id", "type": "uint256"},
        {"indexed": True, "internalType": "address", "name": "user", "type": "address"},
        {"indexed": False, "internalType": "uint256", "name": "refund", "type": "uint256"},
        {"indexed": False, "internalType": "uint256", "name": "penalty", "type": "uint256"},
        {"indexed": False, "internalType": "uint16", "name": "trustScore", "type": "uint16"}],
     "name": "NoShowFinalized", "type": "event"}
]

In [5]:
RPC_URL = "https://json-rpc.evm.testnet.iota.cafe"
w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={"timeout": 30}))
assert w3.is_connected(), "RPC connection failed"
assert w3.eth.chain_id == CHAIN_ID, f"Unexpected chain ID: {w3.eth.chain_id}"

contract_address = Web3.to_checksum_address(SMART_CONTRACT_ADDRESS)
contract = w3.eth.contract(address=contract_address, abi=ABI)

def make_account(role, private_key):
    account = w3.eth.account.from_key(private_key)
    return {"role": role, "address": Web3.to_checksum_address(account.address), "key": private_key}

client = make_account("client", "c150364d9d8633a5af50b41389b1ae388c4f3029367cee65ef289026d7d9a9a9")
oracle = make_account("authorized_oracle", "cb0a6bc96d75c0b80f2d0b5e6f037f6893253284e0646488e1a591d4c4225fa0")
attacker = make_account("unauthorized_user", "93f86ad1b2adc4b2c32b4dfc31e1f7c147c8ca66eb430809553aa011beb2d043")

print("Connected:", w3.is_connected(), "| Chain:", w3.eth.chain_id)
print("Contract:", contract_address)
print("Client:", client["address"])
print("Oracle:", oracle["address"])
print("Attacker:", attacker["address"])


Connected: True | Chain: 1076
Contract: 0x645aDd21578B9fc69Cfbe38bf18F2698Bb449129
Client: 0xC40f4633b305b90cC0A048dfBF16BAf75Eb1b0C8
Oracle: 0x0Fc6140AF8B71795439925543cE481f469196250
Attacker: 0xaf2C57E2bb12e32ad1EFfFc83D3171CcEd51A850


## 3. Evaluation helpers

In [6]:
test_results = []
tx_records = []
slot_cursor = 0

def next_slot():
    global slot_cursor
    if slot_cursor >= len(TEST_SLOTS):
        raise RuntimeError("Add more independent valid slots to TEST_SLOTS")
    slot = TEST_SLOTS[slot_cursor]
    slot_cursor += 1
    return slot

def send_tx(scenario, action, sender, function_call, value=0):
    nonce = w3.eth.get_transaction_count(sender["address"], "pending")
    gas_price = w3.eth.gas_price
    tx = function_call.build_transaction({
        "from": sender["address"], "value": value, "nonce": nonce,
        "gas": GAS_LIMIT, "gasPrice": gas_price, "chainId": CHAIN_ID})
    signed = w3.eth.account.sign_transaction(tx, sender["key"])
    raw = getattr(signed, "raw_transaction", signed.rawTransaction if hasattr(signed, "rawTransaction") else None)
    started = time.perf_counter()
    tx_hash = w3.eth.send_raw_transaction(raw)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=RECEIPT_TIMEOUT)
    latency = time.perf_counter() - started
    effective_price = int(receipt.get("effectiveGasPrice", gas_price))
    gas_used = int(receipt["gasUsed"])
    cost_wei = gas_used * effective_price
    record = {
        "scenario": scenario, "action": action, "sender_role": sender["role"],
        "tx_hash": tx_hash.hex(), "receipt_status": int(receipt["status"]),
        "outcome": "SUCCESS" if receipt["status"] == 1 else "REVERTED",
        "block_number": int(receipt["blockNumber"]), "latency_s": round(latency, 4),
        "gas_used": gas_used, "effective_gas_price_wei": effective_price,
        "cost_wei": cost_wei, "cost_native": str(Decimal(cost_wei) / Decimal(10**18))}
    tx_records.append(record)
    print(f"[{scenario}] {action}: {record['outcome']} | gas={gas_used} | latency={latency:.3f}s")
    return record, receipt

def add_result(scenario, condition, expected, observed, verdict, final_state, txs=None):
    test_results.append({
        "scenario": scenario, "injected_condition": condition,
        "expected_behavior": expected, "observed_result": observed,
        "verdict": verdict, "final_state": final_state,
        "tx_hashes": ";".join(t["tx_hash"] for t in (txs or []))})

def reserve(scenario, slot, sender=None, value=RESERVE_VALUE_WEI):
    sender = sender or client
    record, receipt = send_tx(scenario, f"reserve({slot})", sender, contract.functions.reserve(slot), value)
    if receipt["status"] != 1:
        raise RuntimeError(f"Setup reservation reverted for {scenario}")
    events = contract.events.ReservationCreated().process_receipt(receipt)
    if not events:
        raise RuntimeError("ReservationCreated event not found")
    event = events[0]["args"]
    return {"id": int(event["id"]), "deadline": int(event["deadline"]), "slot": slot, "tx": record}

def wait_after(deadline, max_wait_seconds=900):
    remaining = deadline - int(time.time()) + 1
    if remaining > max_wait_seconds:
        return False
    while int(time.time()) <= deadline:
        time.sleep(min(2, max(0.1, deadline - time.time() + 0.1)))
    return True

## 4. Core logical and malicious-user tests

These tests use separate slots to avoid interference between scenarios. Re-run them only after deploying a fresh evaluation contract or resetting its state.

In [7]:
candidate_amounts = [
    0.1,
    0.2,
    0.5,
    1,
    2,
    5,
    10,
]

slot = "P1"

for amount in candidate_amounts:
    value = Web3.to_wei(amount, "ether")

    try:
        reservation_id = contract.functions.reserve(slot).call({
            "from": client["address"],
            "value": value,
        })

        print(f"SUCCESS with {amount} native tokens")
        print("Returned reservation ID:", reservation_id)
        break

    except Exception as error:
        message = str(error)

        if "payment too low" in message:
            print(f"{amount}: payment too low")
        else:
            print(f"{amount}: another rejection")
            print(message)
            break

0.1: another rejection
('execution reverted: slot reserved', '0x08c379a00000000000000000000000000000000000000000000000000000000000000020000000000000000000000000000000000000000000000000000000000000000d736c6f7420726573657276656400000000000000000000000000000000000000')


In [8]:
# A. Valid reservation (baseline gas/cost)
r = reserve("Valid reservation", next_slot())
add_result("Valid reservation", "Valid paid reservation", "Accepted",
           f"SUCCESS; id={r['id']}", "PASS", "Reservation active", [r["tx"]])

# B. Double reservation
slot = next_slot()
first = reserve("Double reservation", slot)
second, _ = send_tx("Double reservation", "second reservation", attacker,
                    contract.functions.reserve(slot), RESERVE_VALUE_WEI)
add_result("Double reservation", "Second account reserves occupied slot", "Second transaction reverts",
           second["outcome"], "PASS" if second["receipt_status"] == 0 else "FAIL",
           "First reservation unchanged" if second["receipt_status"] == 0 else "Conflict accepted",
           [first["tx"], second])

# C. Early no-show
r = reserve("Early no-show", next_slot())
early, _ = send_tx("Early no-show", "finalize before deadline", client,
                   contract.functions.finalizeNoShow(r["id"]))
add_result("Early no-show", "No-show before deadline", "Transaction reverts", early["outcome"],
           "PASS" if early["receipt_status"] == 0 else "FAIL",
           "Reservation/escrow unchanged" if early["receipt_status"] == 0 else "Premature penalty",
           [r["tx"], early])

# D. Nonexistent reservation ID
invalid, _ = send_tx("Invalid reservation ID", "finalize nonexistent reservation", attacker,
                     contract.functions.finalizeNoShow(NONEXISTENT_ID))
add_result("Invalid reservation ID", "Malicious call with nonexistent ID", "Transaction reverts",
           invalid["outcome"], "PASS" if invalid["receipt_status"] == 0 else "FAIL",
           "No state change" if invalid["receipt_status"] == 0 else "Invalid ID accepted", [invalid])

# E. Insufficient payment
slot = next_slot()
unpaid, _ = send_tx("Insufficient payment", "zero-value reservation", attacker,
                    contract.functions.reserve(slot), 0)
add_result("Insufficient payment", "Reservation without payment", "Transaction reverts",
           unpaid["outcome"], "PASS" if unpaid["receipt_status"] == 0 else "FAIL",
           "Slot remains free" if unpaid["receipt_status"] == 0 else "Unpaid reservation accepted", [unpaid])

[Valid reservation] reserve(T1): SUCCESS | gas=211941 | latency=2.183s


c:\Users\chaym\miniconda3\envs\parking\lib\site-packages\web3\contract\base_contract.py:188: UserWarning: The log with transaction hash: HexBytes('0x42e13039c9d53fb9bc07719b5ab12d75b576ca5d5b5dae17aff39afc013f41ba') and logIndex: 0 encountered the following error during processing: MismatchedABI(The event signature did not match the provided ABI). It has been discarded.
  warnings.warn(


[Double reservation] reserve(T2): SUCCESS | gas=211941 | latency=2.381s


c:\Users\chaym\miniconda3\envs\parking\lib\site-packages\web3\contract\base_contract.py:188: UserWarning: The log with transaction hash: HexBytes('0x3a8b3c2ae7e5f56c345005b6550847ab9bf391dde5233a82b0ee18f8647c7d1f') and logIndex: 0 encountered the following error during processing: MismatchedABI(The event signature did not match the provided ABI). It has been discarded.
  warnings.warn(


[Double reservation] second reservation: REVERTED | gas=22812 | latency=2.299s
[Early no-show] reserve(T3): SUCCESS | gas=211941 | latency=2.467s


c:\Users\chaym\miniconda3\envs\parking\lib\site-packages\web3\contract\base_contract.py:188: UserWarning: The log with transaction hash: HexBytes('0xc683aaa0c2043bef277e0b23c79b621fa9a576fc25829dad7755e6237753c6dd') and logIndex: 0 encountered the following error during processing: MismatchedABI(The event signature did not match the provided ABI). It has been discarded.
  warnings.warn(


[Early no-show] finalize before deadline: REVERTED | gas=22479 | latency=2.205s
[Invalid reservation ID] finalize nonexistent reservation: REVERTED | gas=22259 | latency=2.153s
[Insufficient payment] zero-value reservation: REVERTED | gas=23380 | latency=2.116s


## 5. Arrival, replay, oracle-compromise, and delayed-message tests

The MQTT broker is off-chain. A delayed MQTT test is therefore represented by delaying the backend's resulting `confirmArrival` transaction until after the reservation deadline.

In [ ]:
# F. Confirmation without reservation
missing, _ = send_tx("Confirmation without reservation", "confirm nonexistent reservation", oracle,
                     contract.functions.confirmArrival(NONEXISTENT_ID))
add_result("Confirmation without reservation", "Oracle confirms nonexistent reservation",
           "Transaction reverts", missing["outcome"],
           "PASS" if missing["receipt_status"] == 0 else "FAIL",
           "No state change" if missing["receipt_status"] == 0 else "False state accepted", [missing])

# G. False arrival by unauthorized account
r = reserve("False arrival", next_slot())
false_arrival, _ = send_tx("False arrival", "unauthorized confirmArrival", attacker,
                           contract.functions.confirmArrival(r["id"]))
add_result("False arrival", "Unauthorized account confirms valid reservation", "Transaction reverts",
           false_arrival["outcome"], "PASS" if false_arrival["receipt_status"] == 0 else "FAIL",
           "Reservation remains unconfirmed" if false_arrival["receipt_status"] == 0 else "False arrival accepted",
           [r["tx"], false_arrival])

# H. Replayed arrival
r = reserve("Replayed arrival", next_slot())
first_confirmation, _ = send_tx("Replayed arrival", "first confirmArrival", oracle,
                                contract.functions.confirmArrival(r["id"]))
replay, _ = send_tx("Replayed arrival", "replayed confirmArrival", oracle,
                    contract.functions.confirmArrival(r["id"]))
replay_pass = first_confirmation["receipt_status"] == 1 and replay["receipt_status"] == 0
add_result("Replayed arrival", "Same authorized confirmation submitted twice",
           "First succeeds; replay reverts",
           f"first={first_confirmation['outcome']}; replay={replay['outcome']}",
           "PASS" if replay_pass else "FAIL", "Single confirmation retained" if replay_pass else "Unexpected state",
           [r["tx"], first_confirmation, replay])

# I. Authorized but compromised oracle
r = reserve("Compromised oracle", next_slot())
forged, _ = send_tx("Compromised oracle", "fabricated authorized confirmation", oracle,
                    contract.functions.confirmArrival(r["id"]))
accepted = forged["receipt_status"] == 1
add_result("Compromised oracle", "Authorized oracle fabricates arrival without physical evidence",
           "Trust-boundary behavior is measured", forged["outcome"],
           "LIMITATION" if accepted else "PASS",
           "Fabricated arrival accepted by trusted-oracle design" if accepted else "Rejected by another invariant",
           [r["tx"], forged])

[Confirmation without reservation] confirm nonexistent reservation: REVERTED | gas=22588 | latency=1.756s
[False arrival] reserve(R5): SUCCESS | gas=211941 | latency=2.433s


c:\Users\chaym\miniconda3\envs\parking\lib\site-packages\web3\contract\base_contract.py:188: UserWarning: The log with transaction hash: HexBytes('0x1baae73be2bbbe0dffe003d37e741a994e57451ef0ed7ac855e744be3d5b31b9') and logIndex: 0 encountered the following error during processing: MismatchedABI(The event signature did not match the provided ABI). It has been discarded.
  warnings.warn(


[False arrival] unauthorized confirmArrival: REVERTED | gas=22257 | latency=2.424s
[Replayed arrival] reserve(R6): SUCCESS | gas=211941 | latency=1.699s


c:\Users\chaym\miniconda3\envs\parking\lib\site-packages\web3\contract\base_contract.py:188: UserWarning: The log with transaction hash: HexBytes('0xa60fad87dea8db3b8c3305c9bb60d8992ee13e0d7626211ad154768e4c717420') and logIndex: 0 encountered the following error during processing: MismatchedABI(The event signature did not match the provided ABI). It has been discarded.
  warnings.warn(


[Replayed arrival] first confirmArrival: SUCCESS | gas=43517 | latency=3.392s
[Replayed arrival] replayed confirmArrival: REVERTED | gas=22588 | latency=2.153s
[Compromised oracle] reserve(R7): SUCCESS | gas=211941 | latency=3.489s


c:\Users\chaym\miniconda3\envs\parking\lib\site-packages\web3\contract\base_contract.py:188: UserWarning: The log with transaction hash: HexBytes('0x40210b73c9e9557a89a385644508a4d1bedf247618a387131474cd0eb934d590') and logIndex: 0 encountered the following error during processing: MismatchedABI(The event signature did not match the provided ABI). It has been discarded.
  warnings.warn(


[Compromised oracle] fabricated authorized confirmation: SUCCESS | gas=39573 | latency=2.243s


In [ ]:
# J. Delayed MQTT-derived arrival -- this cell may wait until the deadline.
r = reserve("Delayed MQTT message", next_slot())
if wait_after(r["deadline"], max_wait_seconds=900):
    delayed, _ = send_tx("Delayed MQTT message", "confirmArrival after deadline", oracle,
                         contract.functions.confirmArrival(r["id"]))
    add_result("Delayed MQTT message", "Backend submits delayed event after deadline",
               "Transaction reverts", delayed["outcome"],
               "PASS" if delayed["receipt_status"] == 0 else "FAIL",
               "Stale event rejected" if delayed["receipt_status"] == 0 else "Stale event accepted",
               [r["tx"], delayed])
else:
    add_result("Delayed MQTT message", "Backend submits delayed event after deadline",
               "Transaction reverts", "Deadline exceeds 900-second notebook limit",
               "SKIP", "Not evaluated", [r["tx"]])

[Delayed MQTT message] reserve(R8): SUCCESS | gas=211941 | latency=1.738s


c:\Users\chaym\miniconda3\envs\parking\lib\site-packages\web3\contract\base_contract.py:188: UserWarning: The log with transaction hash: HexBytes('0x579d14218a904f070c52d94b01b1c7e39fa9cc64472c00096b051fff3b65cdc3') and logIndex: 0 encountered the following error during processing: MismatchedABI(The event signature did not match the provided ABI). It has been discarded.
  warnings.warn(


[Delayed MQTT message] confirmArrival after deadline: SUCCESS | gas=39573 | latency=2.574s


In [ ]:
# K. No-show dispute after valid arrival -- this cell may wait until the deadline.
r = reserve("No-show dispute", next_slot())
confirmed, _ = send_tx("No-show dispute", "valid confirmArrival", oracle,
                       contract.functions.confirmArrival(r["id"]))
if confirmed["receipt_status"] != 1:
    add_result("No-show dispute", "No-show after valid arrival", "Confirmation succeeds first",
               confirmed["outcome"], "FAIL", "Arrival was not confirmed", [r["tx"], confirmed])
elif wait_after(r["deadline"], max_wait_seconds=900):
    disputed, _ = send_tx("No-show dispute", "finalizeNoShow after confirmation", client,
                          contract.functions.finalizeNoShow(r["id"]))
    add_result("No-show dispute", "No-show requested after confirmed arrival", "Transaction reverts",
               disputed["outcome"], "PASS" if disputed["receipt_status"] == 0 else "FAIL",
               "Confirmed state unchanged" if disputed["receipt_status"] == 0 else "Confirmed user penalized",
               [r["tx"], confirmed, disputed])
else:
    add_result("No-show dispute", "No-show requested after confirmed arrival", "Transaction reverts",
               "Deadline exceeds 900-second notebook limit", "SKIP", "Not evaluated", [r["tx"], confirmed])

[No-show dispute] reserve(R9): SUCCESS | gas=211941 | latency=2.393s


c:\Users\chaym\miniconda3\envs\parking\lib\site-packages\web3\contract\base_contract.py:188: UserWarning: The log with transaction hash: HexBytes('0xda8a6f0d24f7a9756f94634cb399e458c7253d0812f514f90cdd476c4d221240') and logIndex: 0 encountered the following error during processing: MismatchedABI(The event signature did not match the provided ABI). It has been discarded.
  warnings.warn(


[No-show dispute] valid confirmArrival: SUCCESS | gas=43517 | latency=2.545s
[No-show dispute] finalizeNoShow after confirmation: REVERTED | gas=22259 | latency=2.392s


## 6. Refund-failure requirement

A normal Web3 account cannot refuse an incoming native-token transfer. A real refund-failure test requires a deployed Solidity receiver whose `receive()`/`fallback()` reverts, plus the ABI and lifecycle of the contract's cancellation/refund function. Until that harness is added, this scenario must be reported as **not evaluated**, not as passed.

In [22]:
add_result(
    "Payment refund failure",
    "Recipient contract rejects the native-token refund",
    "Operation fails safely and accounting remains consistent",
    "Rejecting-receiver harness and refund/cancel ABI are required",
    "SKIP",
    "Not evaluated")

## 7. Results and exports

`PASS` means the observed on-chain behavior matched the expected invariant. `FAIL` means it did not. `LIMITATION` identifies a measured trust-model weakness, and `SKIP` identifies a scenario that was not experimentally executed.

In [23]:
results_df = pd.DataFrame(test_results)
gas_df = pd.DataFrame(tx_records)

display(results_df[["scenario", "expected_behavior", "observed_result", "verdict", "final_state"]])
display(gas_df[["scenario", "action", "outcome", "gas_used", "effective_gas_price_wei", "cost_native", "latency_s"]])

output_dir = Path("contract_evaluation_results")
output_dir.mkdir(exist_ok=True)
results_df.to_csv(output_dir / "robustness_test_matrix.csv", index=False)
gas_df.to_csv(output_dir / "transaction_gas_breakdown.csv", index=False)
print("Saved:", output_dir.resolve())
print(results_df["verdict"].value_counts(dropna=False))

,scenario,expected_behavior,observed_result,verdict,final_state
0,Valid reservation,Accepted,SUCCESS; id=11,PASS,Reservation active
1,Double reservation,Second transaction reverts,REVERTED,PASS,First reservation unchanged
2,Early no-show,Transaction reverts,SUCCESS,FAIL,Premature penalty
3,Invalid reservation ID,Transaction reverts,REVERTED,PASS,No state change
4,Insufficient payment,Transaction reverts,REVERTED,PASS,Slot remains free
5,Confirmation without reservation,Transaction reverts,REVERTED,PASS,No state change
6,False arrival,Transaction reverts,REVERTED,PASS,Reservation remains unconfirmed
7,Replayed arrival,First succeeds; replay reverts,first=SUCCESS; replay=REVERTED,PASS,Single confirmation retained
8,Compromised oracle,Trust-boundary behavior is measured,SUCCESS,LIMITATION,Fabricated arrival accepted by trusted-oracle ...
9,Delayed MQTT message,Transaction reverts,SUCCESS,FAIL,Stale event accepted


,scenario,action,outcome,gas_used,effective_gas_price_wei,cost_native,latency_s
0,Valid reservation,reserve(R1),SUCCESS,211941,10000000000,0.00211941,2.4740
1,Double reservation,reserve(R2),SUCCESS,211941,10000000000,0.00211941,1.8414
2,Double reservation,second reservation,REVERTED,22812,10000000000,0.00022812,2.0087
3,Early no-show,reserve(R3),SUCCESS,211941,10000000000,0.00211941,2.6835
4,Early no-show,finalize before deadline,SUCCESS,51192,10000000000,0.00051192,2.7110
5,Invalid reservation ID,finalize nonexistent reservation,REVERTED,22259,10000000000,0.00022259,2.2268
6,Insufficient payment,zero-value reservation,REVERTED,23380,10000000000,0.0002338,1.8984
7,Confirmation without reservation,confirm nonexistent reservation,REVERTED,22588,10000000000,0.00022588,1.7556
8,False arrival,reserve(R5),SUCCESS,211941,10000000000,0.00211941,2.4327
9,False arrival,unauthorized confirmArrival,REVERTED,22257,10000000000,0.00022257,2.4238


Saved: D:\Parking\contract_evaluation_results
verdict
PASS          8
FAIL          2
LIMITATION    1
SKIP          1
Name: count, dtype: int64


In [26]:
gas_table = gas_df[
    [
        "scenario",
        "action",
        "outcome",
        "gas_used",
        "effective_gas_price_wei",
        "cost_native",
        "latency_s",
    ]
].copy()

display(gas_table)

,scenario,action,outcome,gas_used,effective_gas_price_wei,cost_native,latency_s
0,Valid reservation,reserve(R1),SUCCESS,211941,10000000000,0.00211941,2.4740
1,Double reservation,reserve(R2),SUCCESS,211941,10000000000,0.00211941,1.8414
2,Double reservation,second reservation,REVERTED,22812,10000000000,0.00022812,2.0087
3,Early no-show,reserve(R3),SUCCESS,211941,10000000000,0.00211941,2.6835
4,Early no-show,finalize before deadline,SUCCESS,51192,10000000000,0.00051192,2.7110
5,Invalid reservation ID,finalize nonexistent reservation,REVERTED,22259,10000000000,0.00022259,2.2268
6,Insufficient payment,zero-value reservation,REVERTED,23380,10000000000,0.0002338,1.8984
7,Confirmation without reservation,confirm nonexistent reservation,REVERTED,22588,10000000000,0.00022588,1.7556
8,False arrival,reserve(R5),SUCCESS,211941,10000000000,0.00211941,2.4327
9,False arrival,unauthorized confirmArrival,REVERTED,22257,10000000000,0.00022257,2.4238
